<div align="center">
  <img src="https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/wsu_logo_horizontal.png" alt="Wayne State University Logo" width="320">
  <h1>Lab 7: Anomaly Detection with Manufacturing Sound</h1>
</div>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WSU-AI-in-ME/ai-in-me-1/blob/main/labs/week08/lab07_normal_only_anomaly_detection.ipynb)

[Course Repository](https://github.com/WSU-AI-in-ME/ai-in-me-1)

[Lab Index](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/Lab_index.ipynb)

### ME 5995: AI in Mechanical Engineering I: Fundamentals of Manufacturing Data Science

**Wayne State University**

**Individual graded Lab**

**Version: Student Notebook**

Open in Colab and save a personal copy before editing.
Canvas controls assignment dates and submission requirements.

## Student Information

Double-click this Markdown cell to edit.

**Student name:** TODO: Enter your name  
**WSU AccessID:** TODO: Enter your WSU AccessID

## Overview and learning objectives

Can a normal sound pattern reveal an anomalous recording from the same fan ID?
Learn a reference from normal data, then measure how well new observations
reconstruct from that reference. This extends Practice 8 from synthetic points
to real recording-level features. Practice completion is not a prerequisite.

By the end, you can:

- identify which rows may fit the scaler and PCA;
- interpret one reconstruction-error score per recording;
- select a threshold from validation evidence, including ties;
- interpret final false alarms and missed anomalies within the evaluation scope.

**Your work:** two setting changes and four short checkpoints, worth 10 points.
Run cells in order. Cells marked **PROVIDED** run unchanged; focus on what
their inputs and outputs mean rather than memorizing the plotting syntax.
Initial settings run successfully but do not complete the required comparisons.

| Checkpoint | Work | Points |
|---|---|---:|
| [CP1](#cp1) | Identify normal fitting rows and their role | 2 |
| [CP2](#cp2) | Inspect a second recording and explain reconstruction error | 2 |
| [CP3](#cp3) | Select a threshold (2) and justify it with validation evidence (2) | 4 |
| [CP4](#cp4) | Interpret frozen test errors and evaluation limits | 2 |

No separate report is required.

## Data and feature context

Use the existing **MIMII public 1.0** course fan `id_06` subset: 50 normal and
20 anomaly recordings. One CSV row is one complete 10-second recording.
Course audio is mono PCM16 at 16 kHz, retaining source microphone Channel 1.
The source 0 dB condition includes mixed factory background noise.

The supplied table has 16 fixed 500-Hz bands covering 0 to 8,000 Hz.
Each value describes log sound-signal power in one band, using a fixed digital
reference, **not Pa or dB SPL**. Feature extraction is already complete;
its window, normalization and floor are documented in the supplied
`LAB07_FEATURE_DICTIONARY.md` for reference.

Labels are file-level: **0 = normal, 1 = anomaly**. They do not identify the
exact abnormal interval or diagnose a particular fault. Features, partition
and task are course-derived, not the official MIMII benchmark protocol.

Only the 16 numeric feature columns enter the model. Labels and other metadata
are excluded. We evaluate other recordings from the same fan ID.

**Software/files:** Colab provides NumPy, pandas, Matplotlib and scikit-learn.
The feature CSV loads directly from the course repository.
See the [feature dictionary](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/mimii/features/lab07/LAB07_FEATURE_DICTIONARY.md)
and [data license](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/mimii/features/lab07/LICENSE_DATA.txt).
No raw-audio download, audio playback or package installation is required.

In [ ]:
# PROVIDED: tables, explicit numerical steps and compact plots.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import ConfusionMatrixDisplay

In [ ]:
# PROVIDED: load the table; data has one row per complete sound recording.
# Load the fixed course feature table directly; no audio download is needed.
feature_csv = 'https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/data/mimii/features/lab07/lab07_features.csv'
data = pd.read_csv(feature_csv)

In [ ]:
# PROVIDED: one input column for each 500-Hz band, in frequency order.
# This explicit list keeps label, split and recording metadata out of the model.
feature_columns = [
    'log_power_0000_0500', 'log_power_0500_1000',
    'log_power_1000_1500', 'log_power_1500_2000',
    'log_power_2000_2500', 'log_power_2500_3000',
    'log_power_3000_3500', 'log_power_3500_4000',
    'log_power_4000_4500', 'log_power_4500_5000',
    'log_power_5000_5500', 'log_power_5500_6000',
    'log_power_6000_6500', 'log_power_6500_7000',
    'log_power_7000_7500', 'log_power_7500_8000']

<a id="cp1"></a>
## CP1: Normal training and split (2 points)

The recording-level split is supplied and fixed. Do not create a new split.
The table below checks the partition/label counts. We will use training to
learn a normal reference, validation to calibrate the threshold, then test
to evaluate the frozen detector. No recording appears in multiple partitions.

In [ ]:
# PROVIDED: crosstab counts recordings for each split/condition combination.
partition_counts = pd.crosstab(data['split'], data['condition'])
# Reorder the display only; no recordings are reassigned to another partition.
partition_counts = partition_counts.reindex(
    index=['training', 'validation', 'test'], columns=['normal', 'anomaly'])
display(partition_counts)

In [ ]:
# PROVIDED: each mask contains one True/False value per recording.
# == compares a value; = assigns the resulting mask to a variable.
training_mask = data['split'] == 'training'
validation_mask = data['split'] == 'validation'
# .loc[mask] keeps only the rows where that mask is True.
training_records = data.loc[training_mask]
validation_records = data.loc[validation_mask]
# Renumber the selected rows from 0; drop=True discards the old row index.
validation_records = validation_records.reset_index(drop=True)

In one or two sentences, identify which rows may fit the scaler/PCA and explain how the other partitions are used.

**Your response:** TODO.

## Guided normal-pattern model: run unchanged

This is the same sequence as Practice 8 C1. StandardScaler learns a mean and
standard deviation for each feature from training. PCA learns two directions
from those standardized training values. `transform` applies a learned object;
it does not refit it. PCA score coordinates are different from anomaly scores.

We fix two PCs for this teaching comparison. Explained variance describes the
retained training variation, not the detector's accuracy or fault probability.

In [ ]:
# PROVIDED: X has recordings in rows and 16 input features in columns.
# table[feature_columns] selects the named feature columns in their listed order.
training_features = training_records[feature_columns]
validation_features = validation_records[feature_columns]
# .to_numpy() gives numerical arrays to scikit-learn, without table labels.
X_training = training_features.to_numpy()
X_validation = validation_features.to_numpy()
# Validation labels are used for calibration only, never as PCA fitting targets.
validation_label_series = validation_records['label']
y_validation = validation_label_series.to_numpy()
print('Normal training feature shape:', X_training.shape)
print('Validation feature shape:', X_validation.shape)

In [ ]:
# PROVIDED: learn feature means/scales from normal training only.
scaler = StandardScaler()
scaler.fit(X_training)
# For each feature: subtract its training mean, divide by its training SD.
# The transformed values are dimensionless; reuse this reference on validation.
X_training_scaled = scaler.transform(X_training)
X_validation_scaled = scaler.transform(X_validation)

In [ ]:
# PROVIDED: compress 16 standardized features into two learned coordinates.
# Keep these settings fixed: two PCs, no rescaling of the PC coordinates.
pca = PCA(n_components=2, whiten=False, svd_solver='full')
pca.fit(X_training_scaled)
# Each row now has two PC coordinates, not an anomaly score yet.
training_coordinates = pca.transform(X_training_scaled)
validation_coordinates = pca.transform(X_validation_scaled)
# Each variance fraction refers to one learned PC of normal training data.
retained_variance = np.sum(pca.explained_variance_ratio_)
print('PC variance fractions:', pca.explained_variance_ratio_)
print('Total retained variance:', retained_variance)

### Reconstruction and per-recording error

`inverse_transform` reconstructs the **16 standardized feature values** from
two PCA coordinates. Subtract, square, then average across feature columns
(`axis=1`). The result is one dimensionless anomaly score per recording.
A large value means poor reconstruction from the learned normal subspace.

Like Practice C1, anomalies along the learned subspace may still reconstruct
well. High retained variance does not guarantee anomaly detection.

In [ ]:
# PROVIDED: complete example of the reconstruction-error calculation.
training_reconstructed = pca.inverse_transform(training_coordinates)
training_difference = X_training_scaled - training_reconstructed
# Square each difference so positive and negative mismatches cannot cancel.
training_squared_error = training_difference ** 2
# axis=1 averages across the 16 columns; axis=0 would average across recordings.
training_error = np.mean(training_squared_error, axis=1)

In [ ]:
# PROVIDED: apply the same calculation without fitting on validation.
validation_reconstructed = pca.inverse_transform(validation_coordinates)
validation_difference = X_validation_scaled - validation_reconstructed
validation_squared_error = validation_difference ** 2
# Keep one score per validation recording, using the same standardized units.
validation_error = np.mean(validation_squared_error, axis=1)

<a id="cp2"></a>
## CP2: Reconstruction evidence (2 points)

Run with the initial `inspection_row = 0`, then change it to **10** and rerun
the inspection, comparison-table and drawing cells. Row 0 is the first normal
validation recording; row 10 is the first anomaly validation recording.
The final table/figure always retains row 0 as a reference, so screenshots of
earlier runs are not needed.

In [ ]:
# TODO: change 0 to 10 after inspecting the initial output.
inspection_row = 0
# Array indices start at 0; [inspection_row] reads one recording's score.
reference_error = validation_error[0]
selected_error = validation_error[inspection_row]
# A two-dimensional array uses [row] to read all 16 features of one recording.
selected_original = X_validation_scaled[inspection_row]
selected_reconstructed = validation_reconstructed[inspection_row]

In [ ]:
# PROVIDED: pair the fixed reference with your selected validation recording.
# Each assigned list supplies two values, one for each table row.
inspection_table = pd.DataFrame()
inspection_table['Role'] = ['Normal reference', 'Selected validation']
# .loc[row, column] reads one table entry, here a recording ID.
inspection_table['Recording ID'] = [validation_records.loc[0, 'recording_id'],
    validation_records.loc[inspection_row, 'recording_id']]
inspection_table['Label'] = [y_validation[0], y_validation[inspection_row]]
inspection_table['Residual score'] = [reference_error, selected_error]
display(inspection_table)

### Provided comparison plot

Read the mismatch between the two curves in each panel. `axes[0]` is the
normal reference on the left; `axes[1]` is your selected recording on the right.
Both use the same y scale. Run this drawing cell unchanged.

In [ ]:
# PROVIDED: arange gives band centers 250, 750, ..., 7750 Hz (stop excluded).
band_centers_hz = np.arange(250, 8000, 500)
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
axes[0].plot(band_centers_hz, X_validation_scaled[0], 'o-', label='Original')
axes[0].plot(band_centers_hz, validation_reconstructed[0], 's--', label='Reconstruction')
axes[1].plot(band_centers_hz, selected_original, 'o-', label='Original')
axes[1].plot(band_centers_hz, selected_reconstructed, 's--', label='Reconstruction')
axes[0].set_title('Normal reference: validation row 0')
axes[1].set_title('Selected validation recording')
axes[0].set_xlabel('Band center frequency (Hz)')
axes[1].set_xlabel('Band center frequency (Hz)')
axes[0].set_ylabel('Standardized feature value (dimensionless)')
axes[0].legend()
axes[1].legend()
fig.tight_layout()
plt.show()

In one sentence, use the comparison to explain what this recording-level residual measures. Cite the two scores approximately.

**Your response:** TODO.

<a id="cp3"></a>
## CP3: Validation threshold (4 points)

Form three candidate thresholds from the 90th, 95th and 99th percentiles of
**training** errors. A recording is predicted anomalous when its score is
**greater than** the threshold. A training percentile does not guarantee a
future false alarm rate, especially with only 30 training recordings.

The provided table compares the candidates using labeled validation data:

1. Choose the highest validation balanced accuracy.
2. If tied, choose fewer validation false positives.
3. If still tied, choose the larger threshold.

Balanced accuracy is the mean of the two class recalls: the fraction of normals
recognized as normal and the fraction of anomalies detected. FP means a normal
recording predicted anomalous; a miss is an anomaly predicted normal.
Ties are valid outcomes; apply the rule when the validation results tie.
Model fitting is normal-only, but this threshold calibration uses labels.

In [ ]:
# PROVIDED: fixed percentile candidates in the same units as reconstruction error.
# For example, the 90th percentile leaves about 90% of training errors below it.
# method='linear' specifies interpolation between the sorted training errors.
threshold_90 = np.percentile(training_error, 90, method='linear')
threshold_95 = np.percentile(training_error, 95, method='linear')
threshold_99 = np.percentile(training_error, 99, method='linear')
# True means predicted anomaly; a score equal to the threshold remains normal.
prediction_90 = validation_error > threshold_90
prediction_95 = validation_error > threshold_95
prediction_99 = validation_error > threshold_99

In [ ]:
# PROVIDED: compare predictions with labels; these calls do not fit a model.
balanced_90 = balanced_accuracy_score(y_validation, prediction_90)
balanced_95 = balanced_accuracy_score(y_validation, prediction_95)
balanced_99 = balanced_accuracy_score(y_validation, prediction_99)
# Matrix rows are actual labels, columns predicted; order is normal=0, anomaly=1.
# Thus [0, 1] is actual normal/predicted anomaly; [1, 0] is the reverse error.
matrix_90 = confusion_matrix(y_validation, prediction_90, labels=[0, 1])
matrix_95 = confusion_matrix(y_validation, prediction_95, labels=[0, 1])
matrix_99 = confusion_matrix(y_validation, prediction_99, labels=[0, 1])

In [ ]:
# PROVIDED: build a small comparison table with explicit named columns.
# Each list follows candidate order: 90th, 95th, then 99th percentile.
candidate_table = pd.DataFrame()
candidate_table['Training percentile'] = [90, 95, 99]
candidate_table['Threshold'] = [threshold_90, threshold_95, threshold_99]
candidate_table['Validation balanced accuracy'] = [balanced_90, balanced_95, balanced_99]
candidate_table['False alarms'] = [matrix_90[0, 1], matrix_95[0, 1], matrix_99[0, 1]]
candidate_table['Missed anomalies'] = [matrix_90[1, 0], matrix_95[1, 0], matrix_99[1, 0]]
display(candidate_table)

In [ ]:
# PROVIDED: label masks separate the groups for display, not for PCA fitting.
validation_normal_mask = y_validation == 0
validation_anomaly_mask = y_validation == 1
validation_rows = np.arange(len(y_validation))

In [ ]:
# PROVIDED: points are recording scores; horizontal lines are candidate cutoffs.
fig, axis = plt.subplots(figsize=(8, 4))
# array[mask] keeps values in the True positions, as in the earlier row selection.
axis.scatter(validation_rows[validation_normal_mask], validation_error[validation_normal_mask],
    c='seagreen', label='Normal')
axis.scatter(validation_rows[validation_anomaly_mask], validation_error[validation_anomaly_mask],
    c='darkorange', label='Anomaly')
axis.axhline(threshold_90, linestyle='--', label='90th percentile')
axis.axhline(threshold_95, linestyle=':', label='95th percentile')
axis.axhline(threshold_99, linestyle='-.', label='99th percentile')
axis.set_xlabel('Validation recording row')
axis.set_ylabel('Mean squared standardized residual')
axis.set_title('Validation evidence: before final test')
axis.legend(fontsize=8)
fig.tight_layout()
plt.show()

### Select using validation, then freeze

Replace the initial candidate below with `threshold_90`, `threshold_95` or
`threshold_99` according to the rule. Complete the response before running
the test cells. Do not change the feature list, component count or split.

In [ ]:
# TODO: choose the named candidate using the validation rule.
selected_threshold = threshold_90
print('Selected threshold:', selected_threshold)

In one or two sentences, cite the candidate validation false alarms/misses and explain why your chosen threshold follows the selection rule. Distinguish a tie-break from evidence of better performance.

**Your response:** TODO.

<a id="cp4"></a>
## CP4: Frozen test interpretation (2 points)

Freeze the choice before evaluating test. Reuse the training-fitted scaler/PCA;
do not refit or select another threshold after seeing these results.
Restart/Run all reproduces the same teaching evaluation, not a newly unseen test.

The test has ten normal and ten anomaly recordings from the same fan ID.
Some course recordings were seen in earlier activities. This test is held out
from this Lab's fitting/selection, not guaranteed untouched across the course.

In [ ]:
# PROVIDED: copy the chosen threshold before any test score is calculated.
frozen_threshold = selected_threshold
# Repeat the earlier row/feature selection pattern for the fixed test partition.
test_mask = data['split'] == 'test'
test_records = data.loc[test_mask]
test_features = test_records[feature_columns]
X_test = test_features.to_numpy()
test_label_series = test_records['label']
y_test = test_label_series.to_numpy()
# Transform with the existing objects; there is no fit call in this section.
X_test_scaled = scaler.transform(X_test)
test_coordinates = pca.transform(X_test_scaled)
test_reconstructed = pca.inverse_transform(test_coordinates)

In [ ]:
# PROVIDED: same per-recording residual calculation used for training/validation.
test_difference = X_test_scaled - test_reconstructed
test_squared_error = test_difference ** 2
test_error = np.mean(test_squared_error, axis=1)
test_prediction = test_error > frozen_threshold
test_matrix = confusion_matrix(y_test, test_prediction, labels=[0, 1])
# Summing True/False values counts the actual normals (True counts as 1).
test_normal_count = np.sum(y_test == 0)
test_false_alarm_count = test_matrix[0, 1]
test_missed_count = test_matrix[1, 0]
# FPR uses actual normals as its denominator, not all test recordings.
test_false_alarm_rate = test_false_alarm_count / test_normal_count

In [ ]:
# PROVIDED: these metrics use anomaly=1 as the positive class.
# zero_division=0 reports zero when a metric's denominator is zero.
test_precision = precision_score(y_test, test_prediction, zero_division=0)
test_recall = recall_score(y_test, test_prediction, zero_division=0)
test_f1 = f1_score(y_test, test_prediction, zero_division=0)
print('Frozen threshold:', frozen_threshold)
print('Normal false alarms:', test_false_alarm_count, 'of', test_normal_count)
print('Missed anomalies:', test_missed_count)
print('Normal false alarm rate:', test_false_alarm_rate)
print('Anomaly precision:', test_precision)
print('Anomaly recall:', test_recall)
print('Anomaly F1:', test_f1)
print('Precision is zero if no anomalies are predicted.')

In [ ]:
# PROVIDED drawing: matrix rows are actual; columns are predicted.
matrix_display = ConfusionMatrixDisplay(test_matrix, display_labels=['Normal', 'Anomaly'])
fig, axis = plt.subplots(figsize=(5, 4))
matrix_display.plot(ax=axis, cmap='Blues', colorbar=False, values_format='d')
axis.set_title('Frozen detector: final test')
fig.tight_layout()
plt.show()

Use two short sentences: report the test false alarms and misses, then state a same-ID/small-sample limitation. Does observing no misses establish that all future anomalies will be found?

**Your response:** TODO.

The artificial test anomaly proportion is 50%, not a field failure rate.
Precision can change with deployment prevalence. Do not infer a specific
physical fault or background-noise cause from a reconstruction error alone.

<a id="optional"></a>
## Optional challenge: a second detector (ungraded)

**Required work ends here.** You may go directly to the submission checklist.
This supplied comparison requires no code changes or written response.

Isolation Forest learns random partitions of the same full standardized input.
It uses normal training only, then its **own** training-percentile candidates
on validation. Negative `score_samples` makes larger values more anomalous.
Compare validation evidence, not the numerical score scales of the two models.
No test comparison, tuning or performance improvement is required.

In [ ]:
# OPTIONAL, PROVIDED: this model uses 16 standardized features, not two PCA scores.
from sklearn.ensemble import IsolationForest
# Fixed settings make the optional comparison reproducible; do not tune them.
if_model = IsolationForest(n_estimators=100, max_samples='auto',
    contamination='auto', random_state=42)
if_model.fit(X_training_scaled)
# The negative sign gives this score the same high-means-anomalous direction as PCA.
if_training_score = -if_model.score_samples(X_training_scaled)
if_validation_score = -if_model.score_samples(X_validation_scaled)

In [ ]:
# OPTIONAL, PROVIDED: calibrate this detector using its own score units.
if_threshold_90 = np.percentile(if_training_score, 90, method='linear')
if_threshold_95 = np.percentile(if_training_score, 95, method='linear')
if_threshold_99 = np.percentile(if_training_score, 99, method='linear')
if_prediction_90 = if_validation_score > if_threshold_90
if_prediction_95 = if_validation_score > if_threshold_95
if_prediction_99 = if_validation_score > if_threshold_99

In [ ]:
# OPTIONAL, PROVIDED: validation-only comparisons; no test calculations.
if_balanced_90 = balanced_accuracy_score(y_validation, if_prediction_90)
if_balanced_95 = balanced_accuracy_score(y_validation, if_prediction_95)
if_balanced_99 = balanced_accuracy_score(y_validation, if_prediction_99)
if_matrix_90 = confusion_matrix(y_validation, if_prediction_90, labels=[0, 1])
if_matrix_95 = confusion_matrix(y_validation, if_prediction_95, labels=[0, 1])
if_matrix_99 = confusion_matrix(y_validation, if_prediction_99, labels=[0, 1])

In [ ]:
# OPTIONAL, PROVIDED: apply the same validation selection rule when reading this table.
if_table = pd.DataFrame()
if_table['Percentile'] = [90, 95, 99]
if_table['Validation balanced accuracy'] = [if_balanced_90, if_balanced_95, if_balanced_99]
if_table['False alarms'] = [if_matrix_90[0, 1], if_matrix_95[0, 1], if_matrix_99[0, 1]]
if_table['Missed anomalies'] = [if_matrix_90[1, 0], if_matrix_95[1, 0], if_matrix_99[1, 0]]
display(if_table)

## Submission checklist

Rendered notebook checkboxes are not clickable. To record completion, edit
this Markdown cell and change `[ ]` to `[x]`.

- [ ] Entered my name and WSU AccessID.
- [ ] Completed CP1~CP4 responses and both required settings.
- [ ] Retained the reference/selected validation comparison, candidate table and test outputs.
- [ ] Restarted the runtime and ran all cells from top to bottom without errors.
- [ ] Saved `Lab07_Firstname_Lastname.ipynb` and `Lab07_Firstname_Lastname.pdf`.
- [ ] Checked that both files show the required work and submitted both through Canvas.

Work individually. Guided and optional material is not separately graded.
No separate report is required. Follow the instructor's PDF export guidance.
PDF export defects receive feedback rather than point deductions; the notebook
is used to assess work obscured by an export. Both files remain required.

## References and data attribution

- Harsh Purohit et al., *MIMII Dataset: Sound Dataset for Malfunctioning Industrial
  Machine Investigation and Inspection*, 2019. [Paper](https://arxiv.org/abs/1909.09347).
- [Original MIMII public 1.0 dataset](https://doi.org/10.5281/zenodo.3384388),
  created by the cited authors at Hitachi, Ltd., CC BY-SA 4.0.
- [Course audio provenance](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/mimii/README.md).
- [PCA documentation](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html).

This teaching table adds fixed frequency-band features and a recording-level
split to the existing course mono subset. Labels are adapted to normal/anomaly.
Preserve original attribution, the change notice and CC BY-SA 4.0 ShareAlike
terms if reusing the derived data. The supplied `LAB07_FEATURE_DICTIONARY.md`
and `LICENSE_DATA.txt` accompany the public feature table.